# Tashkent House Price Predictor — final demo
This short notebook works in a fresh Google Colab runtime. It obtains the repository, installs dependencies, loads the saved preprocessing/model pipeline, validates raw input, and predicts a historical 2019 listing price in USD.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = 'https://github.com/DilnuraHamdamova/tashkent-house-price-predictor.git'
if not Path('src').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, 'tashkent-project'], check=True)
    os.chdir('tashkent-project')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Working directory:', Path.cwd())

## Load the complete pipeline
The joblib artifact contains both preprocessing and the selected Random Forest, so inference cannot accidentally use notebook-only transformations.

In [ ]:
from src.model import load_artifact, predict_price

artifact = load_artifact('artifacts/house_price_pipeline.joblib')
artifact['metadata']['model_name'], artifact['metadata']['protected_test_comparison']

## Predict one unseen raw example
Edit the values and rerun this cell. District spelling should match one of the documented training districts.

In [ ]:
example = {
    'district': 'Chilonzor',
    'size': 70,
    'rooms': 3,
    'level': 3,
    'max_levels': 5,
    'lat': 41.3002,
    'lng': 69.2108,
}
price, warnings = predict_price(artifact, **example)
print(f'Estimated 2019 listing price: ${price:,.0f} USD')
print('Warnings:', warnings or 'none')

## Validation example
Impossible floor combinations fail clearly instead of silently producing a number.

In [ ]:
try:
    predict_price(artifact, **{**example, 'level': 9, 'max_levels': 5})
except ValueError as error:
    print('Expected validation error:', error)

**Responsible-use note:** This is an educational estimator trained on 2019 asking prices. It is not a current appraisal or a basis for lending, taxation, or legal/financial decisions. Consult recent comparable listings and a qualified professional.